# Evaluating LLM outputs with LLM-as-a-Judge

As soon as you need to evaluate your LLM's outputs, it becomes slightly less fun. Outputs change from run to run, you need to separate takeaways from prose, and while you can eyeball 5-6 outputs, this totally doesn't scale. In this notebook, we'll take a peek at how to use LLMs to grade the outputs of other LLMs.

## Setting things up

The notebook requires your
[Nebius Token Factory](https://tokenfactory.nebius.com) API key:

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("NEBIUS_API_KEY") and os.path.exists("token-factory-key"):
    os.environ["NEBIUS_API_KEY"] = open("token-factory-key", encoding="utf-8").read().strip()

assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in .env"
print("Keys loaded.")

Keys loaded.


# The data

We'll be scoring LLM summaries of the paper
[**AI Adoption in S&P 500 Firms**](https://arxiv.org/abs/2607.08920) by Yang Yu,
Martin Fleming, Lucy Hampton, Christophe Combemale and Neil Thompson (MIT
FutureTech, July 2026).

The question it asks is whether large American companies are actually using AI or merely talking about it. Talk is easy to find and hard to trust, so the authors go looking somewhere companies are legally exposed: the annual report every US public company files with the securities regulator, known as a 10-K.

They took the 500 largest US public companies, pulled every paragraph mentioning AI out of those filings for 2016 to 2025, and rated each company in each year on a scale of 1 to 5 — 1 meaning AI is never mentioned, 5 meaning the company describes AI as central to what it sells and how it operates. Then they lined those ratings up against each company's profits and stock-market valuation.

What they found, in short: in 2025, 11% of these companies rated a 5 and another 10% rated a 4, so 21% in total, up from 5% in 2022. Technology companies account for most of that. Profit margins dip for companies in the middle of adopting and rise for the ones that finish, which is the paper's headline.

The cell below downloads the HTML version and strips the tags.

In [2]:
import html
import re
import urllib.request

PAPER_URL = "https://arxiv.org/html/2607.08920v1"


def strip_html(raw):
    # Drop scripts and styles. Keep the plain-TeX copy of each formula, which
    # arXiv stores in the `alttext` attribute, then remove every other tag.
    raw = re.sub(r"(?is)<(script|style|nav|footer)[^>]*>.*?</\1>", " ", raw)
    raw = re.sub(r"(?is)<math[^>]*alttext=\"([^\"]*)\"[^>]*>.*?</math>", r" \1 ", raw)
    raw = re.sub(r"(?is)</(p|div|li|tr|h[1-6]|section|figcaption|td|th)>", "\n", raw)
    raw = re.sub(r"(?s)<[^>]+>", " ", raw)
    raw = html.unescape(raw)
    raw = re.sub(r"[ \t]+", " ", raw)
    return "\n".join(line.strip() for line in raw.splitlines() if line.strip())


request = urllib.request.Request(PAPER_URL, headers={"User-Agent": "Mozilla/5.0 (course notebook)"})
ARTICLE = strip_html(urllib.request.urlopen(request, timeout=60).read().decode("utf-8", "replace"))

print(f"{len(ARTICLE.split()):,} words, roughly {len(ARTICLE) // 4:,} tokens")

15,715 words, roughly 25,512 tokens


Each of the three models writes `N_REPORTS` summaries of the article. Every
summary comes from a single chat completion, with no tools and no agent loop
involved.

The prompt lives in a variable because it is an input to everything that
follows. If you change its wording, every summary changes with it, and the
verdicts you collected before the change describe a different experiment.

We'll take three models - **nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B**, **nvidia/Nemotron-3_5-Lightning**, and **nvidia/nemotron-3-super-120b-a12b** - and generate `N_REPORTS` summaries with each of them. Repetition will help us mitigate the stochasticity of generation.

We keep `N_REPORTS` small for now to avoid spending too much money.

In [3]:
N_REPORTS = 3   # summaries per model

In [4]:
import time

from openai import OpenAI

client = OpenAI(base_url="https://api.tokenfactory.nebius.com/v1/",
                api_key=os.environ["NEBIUS_API_KEY"])

WRITERS = {
    "nano":      "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    "lightning": "nvidia/Nemotron-3_5-Lightning",
    "super":     "nvidia/nemotron-3-super-120b-a12b",
}

SUMMARY_PROMPT = """Summarize the following article for a business audience in about 250 words.

Cover what was measured, how it was measured, and what the results were. Be
specific about numbers.

--- ARTICLE ---
{article}
"""

reports = {}   # (writer, run) -> text
start = time.time()
for name, model in WRITERS.items():
    for run in range(N_REPORTS):
        answer = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": SUMMARY_PROMPT.format(article=ARTICLE)}],
        )
        reports[(name, run)] = answer.choices[0].message.content or ""
    print(f"{name} wrote {N_REPORTS} summaries")

print(f"\nThat is {len(reports)} summaries in {time.time() - start:.0f}s.")
print("\n--- one of them ---\n")
print(reports[("super", 0)][:700])

nano wrote 3 summaries
lightning wrote 3 summaries
super wrote 3 summaries

That is 9 summaries in 41s.

--- one of them ---



**AI Adoption in S&P 500 Firms (2016‑2025): What Was Measured, How, and Key Findings**

The study measured the depth of AI integration in large U.S. corporations by analysing their annual SEC 10‑K filings. A two‑step process first pulled all paragraphs containing AI‑related keywords (e.g., “artificial intelligence,” “machine learning,” “deep learning”) and then fed those excerpts to a language model (GPT‑5‑mini) together with a five‑point rubric:

1. No mention of AI.  
2. Exploring AI or building capacity.  
3. AI used in specific products/processes without financial emphasis.  
4. AI deployed at production level with explicit cost‑savings or revenue expectations.  
5. AI a core strategic


In the rest of the notebook, we'll do some basic fact-checking for the summaries the models produced - first, without LLMs, then with LLM-as-a-Judge.

# Step 1. Rubric items a regular expression can (probably) settle

Instead of asking whether a summary is good, we ask whether it contains something
the article actually says. Here are three such things:

1. 11% of S&P 500 firms scored 5, meaning deep integration, in 2025
2. 21% scored 4 or 5
3. adoption stood at 5% in 2022

These give us binary checks — "does the summary have it, yes or no" — which are
**rubric items**, and they are far more useful than "score this summary out of
10", because the answer can be checked against the article.

Each is a single number with a single meaning, so a pattern match can look for it.
Here is exactly what the three patterns below do:

| item | pattern | what it matches |
|---|---|---|
| 11% deeply integrated | `\b11%` | the characters `11%`, anywhere in the text |
| 21% scored 4 or 5 | `\b21%` | the characters `21%`, anywhere |
| 5% in 2022 | `from (just )?5%\|5% in 2022` | the phrase "from 5%", "from just 5%", or "5% in 2022" |

Before any of that, the text has to be normalised. Models do not write plain
ASCII: they put a narrow no-break space (U+202F) between the number and the
percent sign, they hyphenate with U+2011 instead of a hyphen, and they capitalise words at the start of a sentence. Those characters look identical on screen and
break a plain `in` test, so `normalise` folds them first.

In [5]:
odd_space = sum(r.count("\u202f") for r in reports.values())
odd_hyphen = sum(r.count("\u2011") for r in reports.values())
print(f"across {len(reports)} summaries: {odd_space} narrow no-break spaces, "
      f"{odd_hyphen} non-breaking hyphens")

across 9 summaries: 190 narrow no-break spaces, 189 non-breaking hyphens


In [6]:
import unicodedata


def normalise(text):
    """Fold the typography models emit, so that matching means something."""
    text = unicodedata.normalize("NFKC", text)         # narrow no-break space -> plain space
    for dash in "‐‑‒–—―−":
        text = text.replace(dash, "-")                 # every unicode dash -> hyphen
    text = re.sub(r"\s+", " ", text)
    return re.sub(r"(\d)\s*%", r"\1%", text).lower()   # "11 %" -> "11%", and lower-case


REGEX_ITEMS = {
    "11% deeply integrated": r"\b11%",
    "21% scored 4 or 5":     r"\b21%",
    "5% in 2022":            r"from (just )?5%|5% in 2022",
}

print(f"{'summary':14s} " + "  ".join(f"{k:22s}" for k in REGEX_ITEMS))
for key, report in sorted(reports.items()):
    flat = normalise(report)
    cells = "  ".join(("yes" if re.search(p, flat) else "no").ljust(22) for p in REGEX_ITEMS.values())
    print(f"{key[0] + '-' + str(key[1]):14s} " + cells)

summary        11% deeply integrated   21% scored 4 or 5       5% in 2022            
lightning-0    no                      yes                     yes                   
lightning-1    yes                     yes                     yes                   
lightning-2    yes                     yes                     yes                   
nano-0         yes                     yes                     no                    
nano-1         yes                     yes                     yes                   
nano-2         yes                     yes                     no                    
super-0        yes                     yes                     yes                   
super-1        yes                     yes                     yes                   
super-2        yes                     yes                     no                    


All three models do well on this test, which is what you would expect: the
numbers are prominent in the article and easy to repeat.

While this might look optimistic, in real practice you won't use regular expressions too often. They are fast and cheap, but also they are brittle and unreliable. Changes as little as `11.0%` instead of `11%` will break them. Also, we only check that `11%` is somewhere in the string - but it may be attributed to something totally different, and we won't see it.

Instead of the presence of characters, we'd better check for meaning. And we'll now do it with LLMs.

## Step 2. Rubric items a regular expression cannot settle

Now, instead of searching for character patterns, let's start checking the actual meaning of a summary.

We'll work with the following three rubrics:

### 1. Are the two numbers attached to the right groups?

The paper has two different numbers. 11% of companies got a rating of 5, meaning
AI is central to the business. 21% got a rating of 4 or 5, meaning AI is either
in production or central. The check is whether the summary puts each number with
the right group.

- Passes: "11% of firms rated 5, and another 10% rated 4, so 21% in total."
- Fails: "21% of firms have deeply integrated AI." 21% is the wider group; deep
  integration is the 11%.

### 2. Does the summary report both halves of the profit result?

The paper's profit finding has two parts. Companies in the middle of adopting AI
have *lower* profit margins. Companies that have finished adopting have *higher*
ones. The check is whether the summary reports both parts.

- Passes: "margins dip for firms partway through adoption and are higher for the
  deepest adopters."
- Fails: "firms with deep AI integration have higher profit margins." True, but it
  drops the dip, which is the part worth knowing.

### 3. Does the summary claim a cause?

The paper reports a correlation. It does not claim that adopting AI *caused* the
higher profits. The check is whether the summary keeps that distinction.

- Passes: "deep adopters show higher margins."
- Fails: "AI delivers profit gains for mature adopters." The word "delivers" says
  AI produced them.

In all three, the passing and failing versions contain the same numbers and
mostly the same words. So whatever we're going to employ to check it, should be able to understand the meaning. Luckily, we have LLMs for that!

## Asking an LLM to judge

Let's ask a model (Nemotron Nano) to score our summaries with respect to the first claim.

The plan is simple:
- give the model a rubric item (what to assess) and the summary,
- tell it what we expect to see as a result (only yes or no),
- enjoy the result:

In [7]:
JUDGE = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B"

ITEM_GROUPS = (
    "Does the summary attach 11% to the firms rated 5 and 21% to the firms rated 4 or 5? "
    "Answer no if it swaps them, or attaches either figure to a different group."
)

ONE_WORD_PROMPT = """You are checking one summary of a research paper against one rubric item.

Rubric item: {item}

Answer with exactly one word, yes or no. No other text.

--- SUMMARY ---
{report}
"""

one_word = {}
for key, report in sorted(reports.items()):
    reply = client.chat.completions.create(
        model=JUDGE,
        messages=[{"role": "user", "content": ONE_WORD_PROMPT.format(item=ITEM_GROUPS,
                                                                     report=report)}],
    )
    one_word[key] = " ".join((reply.choices[0].message.content or "").split())

for key, answer in one_word.items():
    print(f"{key[0] + '-' + str(key[1]):14s} {answer[:60]!r}")

obedient = sum(a.strip().lower().rstrip(".") in {"yes", "no"} for a in one_word.values())
print()
print(f"{obedient} of {len(one_word)} replies were exactly yes or no.")

lightning-0    'no'
lightning-1    'yes'
lightning-2    'yes'
nano-0         'yes'
nano-1         'yes'
nano-2         'yes'
super-0        'yes'
super-1        'yes'
super-2        'yes'

9 of 9 replies were exactly yes or no.


This approach isn't ideal. First of all, it doesn't give you the reason why the model chose *yes* or *no*, and this makes debugging trickier.

We can relieve the requirement of giving a one-word answer - but then we'll need to extract yes or no from the model's reasoning, which isn't always fun.

To make extraction easier, it's always better to use **structured outputs**, which bind the LLM to produce a JSON with a prescribed schema.

Let's describe the answer as a `pydantic` class with two fields:
* `verdict`, which can only be `"yes"` or `"no"` and
* `evidence` which can be an arbitrary string (even an empty one)

Actually, for rubrics 2 and 3 we'll need the third answer option - `not_discussed`. For example, in rubric 3 it's useful to distinguish between three situations:
* A summary correctly mentions correlation,
* A summary invents a causal connection,
* A summary avoids touching the topic at all.

In [9]:
!pip install -q langchain-nebius

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 7.1 MB/s eta 0:00:00


In [10]:
from typing import Literal

from langchain_nebius import ChatNebius
from pydantic import BaseModel, Field


class YesNo(BaseModel):
    verdict: Literal["yes", "no"] = Field(
        description="yes if the summary does what the rubric item asks, no if it does not."
    )
    evidence: str = Field(
        description="The wording from the summary behind this answer."
    )


class YesNoOrMissing(BaseModel):
    verdict: Literal["yes", "no", "not_discussed"] = Field(
        description=("yes if the summary does what the rubric item asks; no if it addresses "
                     "the subject and gets it wrong; not_discussed if the summary never "
                     "raises the subject at all.")
    )
    evidence: str = Field(
        description="The wording from the summary behind this answer, or 'nothing' if the "
                    "summary never raises the subject."
    )


ITEM_DIP = (
    "Does the summary report both halves of the profit finding, that margins are LOWER for "
    "firms partway through adoption and HIGHER for the firms with the deepest adoption? "
    "Answer yes if both halves are there, no if only one of them is, and not_discussed if "
    "the summary never mentions the profit finding."
)

ITEM_CAUSE = (
    "Does the summary present the link between AI adoption and profits as an association "
    "rather than a cause? Answer yes if it keeps it an association, no if it says or implies "
    "that adopting AI CAUSED the higher profits, and not_discussed if the summary never "
    "discusses the link."
)

RUBRIC = {
    "groups kept apart":      (ITEM_GROUPS, YesNo),
    "the dip and the rise":   (ITEM_DIP, YesNoOrMissing),
    "association, not cause": (ITEM_CAUSE, YesNoOrMissing),
}

ITEM_PROMPT = """You are checking one summary of a research paper against one rubric item.

Rubric item: {item}

Answer for the summary below.

--- SUMMARY ---
{report}
"""

base_judge = ChatNebius(model=JUDGE, temperature=0.0, timeout=180)

graded = {}
for key, report in sorted(reports.items()):
    graded[key] = {}
    for label, (item, schema) in RUBRIC.items():
        judge = base_judge.with_structured_output(schema)
        v = judge.invoke(ITEM_PROMPT.format(item=item, report=report))
        graded[key][label] = {"verdict": "no answer" if v is None else v.verdict,
                              "evidence": "" if v is None else v.evidence}

print(f"{'summary':14s} " + "  ".join(f"{k:24s}" for k in RUBRIC))
for key, row in graded.items():
    print(f"{key[0] + '-' + str(key[1]):14s} " + "  ".join(f"{row[k]['verdict']:24s}" for k in RUBRIC))

summary        groups kept apart         the dip and the rise      association, not cause  
lightning-0    no                        yes                       no                      
lightning-1    yes                       yes                       yes                     
lightning-2    yes                       yes                       yes                     
nano-0         yes                       yes                       yes                     
nano-1         yes                       yes                       no answer               
nano-2         yes                       yes                       yes                     
super-0        yes                       yes                       no                      
super-1        yes                       yes                       no                      
super-2        yes                       yes                       no                      


In [11]:
print(f"{'writer':10s} " + "  ".join(f"{k:32s}" for k in RUBRIC))
for name in WRITERS:
    cells = []
    for label in RUBRIC:
        verdicts = [graded[(name, r)][label]["verdict"] for r in range(N_REPORTS)]
        parts = [f"{verdicts.count('yes')} yes", f"{verdicts.count('no')} no"]
        if verdicts.count("not_discussed"):
            parts.append(f"{verdicts.count('not_discussed')} not discussed")
        cells.append(", ".join(parts).ljust(32))
    print(f"{name:10s} " + "  ".join(cells))

writer     groups kept apart                 the dip and the rise              association, not cause          
nano       3 yes, 0 no                       3 yes, 0 no                       2 yes, 0 no                     
lightning  2 yes, 1 no                       3 yes, 0 no                       2 yes, 1 no                     
super      3 yes, 0 no                       3 yes, 0 no                       0 yes, 3 no                     


Quite interestingly, larger models tend to score less at the 3rd rubric.

The next cell prints the explanation for every `no` (the `evidence` field).

In [12]:
for key, row in graded.items():
    for label, cell in row.items():
        if cell["verdict"] == "no":
            print(f"{key[0]}-{key[1]}  [{label}]")
            print(f"   {cell['evidence'][:160]}")

lightning-0  [groups kept apart]
   Deep integration alone rose from 1% to 5.7% over the same period.
lightning-0  [association, not cause]
   Firms moving from no adoption to deep integration saw net profit margins rise by approximately 6 percentage points on average.
super-0  [association, not cause]
   delivering profitability gains
super-1  [association, not cause]
   delivering clear profitability gains for mature adopters
super-2  [association, not cause]
   Deep integrators (score 5) enjoyed margins ≈12.6 pp higher (≈15 % higher after firm‑ and year‑fixed effects) in non‑tech firms, while tech firms saw a more mode


Analyze the provided evidence. Does it seem convincing enough for you?

### Practice

1. Write a prompt for checking that the summary correctly states that "*The technology sector accounts for two-thirds of deeply integrated adoption*". Run the checks. How well do the models score?
2. In the example above, we asked an LLM judge to provide evidence. In theory, it can be useful, because it allows not to reread the whole summary during debugging. In practice, fragments with no context might be hard to analyze; also smaller models aren't guaranteed to provide good evidence. So, you might also want to see the model's **justification** for its verdict. Try adding it to the pydantic scheme in the examples above. Will it make judge's verdicts more transparent in `no` cases?
3. Break a summary and see whether the rubric catches it. Corrupt one of the core facts the rubrics check. Ask an LLM to rewrite a summary as a longer and obscure text or add fictional facts without corrupting any of the rubrics. How would our LLM judge cope with this?

# What next

In the next notebook, we'll use LLM judges to evaluate the deep research pipeline we've built in Module 3.

## A caveat: who judges the judge?

While LLM-as-a-Judge is a popular and effective technique, it's not 100% failure-proof. Judges too get confused, especially if you ask a small model to score something complicated. In the next notebook, you'll see some examples. At the same time, collecting ground truth manually might be difficult, so in some cases assessing the judge will be trickier than assessing the original agent.

The less trivial your check is, the more important good prompting becomes. If you need to judge style, giving examples of good and bad style becomes a neccesity. In the trickiest cases, you might need expert advice to make a good prompt for a judge, and some startups, like **Toloka**, provide such assistance as a service.

